# Fine-tune floorplan YOLO-seg (Colab / Kaggle free GPU)

Train **room / wall / door / window** segmentation for [home-planner](https://github.com/linancemelo/home-planner).

| | |
|---|---|
| Base | `JessiP23/floorplan-seg-v2` (YOLO11s-seg) or Ultralytics `yolo11s-seg.pt` |
| Data | CubiCasa5K via HF `phungpx/cubicassa5k-coco` (**CC BY-NC 4.0**) |
| Output | `best.pt` → copy to `backend/models/floorplan-rw-seg.pt` |
| Local install | `python backend/scripts/install_finetuned_weight.py best.pt` |

**Colab:** Runtime → Change runtime type → **T4 GPU**.  
**Kaggle:** Settings → Accelerator → **GPU**.

預計時間：子集 800 圖 × 30 epochs ≈ 1.5–3 h（T4）。煙測合成資料 ≈ 數分鐘。


In [ ]:
# ===== 0) Config (edit me) =====
PLATFORM = "colab"  # "colab" | "kaggle" | "local"
EPOCHS = 30
IMGSZ = 640          # raised to 896 automatically if VRAM allows
BATCH = 8            # lowered automatically on OOM
MAX_TRAIN = 800      # CubiCasa train cap; set -1 for all (~4k, slower)
MAX_VAL = 80
USE_CUBICASA = True  # False → synthetic smoke only
USE_WEAK_LABELS = True  # pseudo-label uploaded *-b.jpg if present
RESUME = False       # True to continue last run
PROJECT = "fp_seg_runs"
RUN_NAME = "yolo11s_cubicasa"

print("config OK", PLATFORM, EPOCHS, IMGSZ)


In [ ]:
# ===== 1) Install deps =====
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "ultralytics>=8.3.0,<8.5",
    "huggingface_hub>=0.26.0",
    "datasets>=2.19.0",
    "opencv-python-headless",
    "Pillow",
    "pyyaml",
)
print("deps installed")


In [ ]:
# ===== 2) GPU check + workdir =====
import torch, shutil
from pathlib import Path

WORK = Path("/content/fp_train") if PLATFORM == "colab" else Path("/kaggle/working/fp_train")
if PLATFORM == "local":
    WORK = Path("./fp_train").resolve()
WORK.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "VRAM_GB", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print("WARNING: no GPU — training will be very slow. Enable T4/P100 in runtime settings.")

# Optional: upload prepare_yolo_dataset.py via Colab Files, or fetch from GitHub raw
PREPARE_SRC_CANDIDATES = [
    Path("/content/prepare_yolo_dataset.py"),
    WORK / "prepare_yolo_dataset.py",
    Path("prepare_yolo_dataset.py"),
]


## 3) Dataset prep script

If you opened this notebook from the repo, `prepare_yolo_dataset.py` is in the same folder — copy it into the workdir.  
Otherwise the next cell **embeds a minimal copy** so the notebook stays self-contained.


In [ ]:
EMBEDDED_PREPARE = '#!/usr/bin/env python3\n"""Build a YOLO-seg dataset (room/wall/door/window/stair/annotation) for free-GPU fine-tune.\n\nPrimary source: Hugging Face ``phungpx/cubicassa5k-coco`` (CubiCasa5K polygons in\nCOCO format). Upstream CubiCasa5K license is **CC BY-NC 4.0** — research / personal\nnon-commercial fine-tune only; review before any commercial use.\n\nOptional: weak (pseudo) labels on marketing/CAD fixtures via a pretrained YOLO-seg.\n\nUsage (Colab / Kaggle / local)::\n\n  python prepare_yolo_dataset.py --out ./fp_yolo --max-train 800 --max-val 80\n  python prepare_yolo_dataset.py --out ./fp_yolo --weak-label-dir ./real-samples \\\\\n      --weak-model ./floorplan-rw-seg.pt\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nimport shutil\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\n# Target taxonomy must match JessiP23/floorplan-seg-v2\nCLASS_NAMES = {\n    0: "room",\n    1: "wall",\n    2: "door",\n    3: "window",\n    4: "stair",\n    5: "annotation",\n}\n\n# CubiCasa COCO category_id → our class id (skip furniture like bed)\nCOCO_TO_OURS: dict[int, int] = {\n    1: 0,  # bathroom → room\n    3: 2,  # door\n    4: 0,  # kitchen → room\n    5: 0,  # room\n    6: 4,  # stairs → stair\n    7: 1,  # wall\n    8: 3,  # window\n}\n\nLICENSE_NOTE = (\n    "CubiCasa5K (via phungpx/cubicassa5k-coco): Creative Commons Attribution-"\n    "NonCommercial 4.0 (CC BY-NC 4.0). Cite Kalervo et al., SCIA 2019. "\n    "Non-commercial / research fine-tune only unless you obtain other rights."\n)\n\n\ndef _poly_to_yolo_seg(\n    segmentation: list[list[float]] | list[float],\n    width: int,\n    height: int,\n) -> list[str]:\n    """COCO polygon(s) → YOLO-seg normalized lines (one poly per returned string body)."""\n    if not segmentation:\n        return []\n    # HF/COCO may store one flat list or list-of-lists\n    polys: list[list[float]]\n    if segmentation and isinstance(segmentation[0], (int, float)):\n        polys = [list(map(float, segmentation))]  # type: ignore[arg-type]\n    else:\n        polys = [list(map(float, p)) for p in segmentation]  # type: ignore[arg-type]\n\n    out: list[str] = []\n    for poly in polys:\n        if len(poly) < 6:\n            continue\n        coords: list[str] = []\n        ok = True\n        for i in range(0, len(poly) - 1, 2):\n            x = poly[i] / float(width)\n            y = poly[i + 1] / float(height)\n            if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):\n                # clamp mild overflow from rasterization\n                x = min(1.0, max(0.0, x))\n                y = min(1.0, max(0.0, y))\n            coords.append(f"{x:.6f}")\n            coords.append(f"{y:.6f}")\n        if len(coords) >= 6 and ok:\n            out.append(" ".join(coords))\n    return out\n\n\ndef _ann_list_from_row(row: dict[str, Any]) -> list[dict[str, Any]]:\n    anns = row.get("annotations")\n    if anns is None:\n        return []\n    # datasets may return list[dict] or columnar dict-of-lists\n    if isinstance(anns, dict):\n        keys = list(anns.keys())\n        n = len(anns[keys[0]]) if keys else 0\n        return [{k: anns[k][i] for k in keys} for i in range(n)]\n    return list(anns)\n\n\ndef write_data_yaml(out_dir: Path) -> Path:\n    yaml_path = out_dir / "data.yaml"\n    names_block = "\\n".join(f"  {i}: {n}" for i, n in CLASS_NAMES.items())\n    yaml_path.write_text(\n        f"# Auto-generated for home-planner floorplan YOLO-seg fine-tune\\n"\n        f"# {LICENSE_NOTE}\\n"\n        f"path: {out_dir.resolve()}\\n"\n        f"train: images/train\\n"\n        f"val: images/val\\n"\n        f"names:\\n{names_block}\\n",\n        encoding="utf-8",\n    )\n    return yaml_path\n\n\ndef export_cubicasa_hf(\n    out_dir: Path,\n    *,\n    max_train: int | None,\n    max_val: int | None,\n    seed: int = 42,\n) -> dict[str, int]:\n    try:\n        from datasets import load_dataset\n    except ImportError as e:\n        raise SystemExit(\n            "Need `datasets` (pip install datasets). On Colab/Kaggle the notebook "\n            "installs it in the first cell."\n        ) from e\n\n    print("Loading phungpx/cubicassa5k-coco from Hugging Face…")\n    print(f"LICENSE: {LICENSE_NOTE}")\n    ds = load_dataset("phungpx/cubicassa5k-coco")\n\n    counts = {"train": 0, "val": 0}\n    rng = random.Random(seed)\n\n    for split_src, split_dst, limit in (\n        ("train", "train", max_train),\n        ("valid", "val", max_val),\n    ):\n        if split_src not in ds:\n            print(f"WARN: split {split_src} missing; skip", file=sys.stderr)\n            continue\n        img_dir = out_dir / "images" / split_dst\n        lbl_dir = out_dir / "labels" / split_dst\n        img_dir.mkdir(parents=True, exist_ok=True)\n        lbl_dir.mkdir(parents=True, exist_ok=True)\n\n        rows = list(range(len(ds[split_src])))\n        rng.shuffle(rows)\n        if limit is not None:\n            rows = rows[: max(0, limit)]\n\n        for idx in rows:\n            row = ds[split_src][idx]\n            image = row["image"]\n            width = int(row.get("width") or image.width)\n            height = int(row.get("height") or image.height)\n            stem = Path(str(row.get("file_name") or f"{split_dst}_{idx:05d}")).stem\n            # avoid collisions\n            stem = f"{split_dst}_{idx:05d}_{stem}"[:120]\n            img_path = img_dir / f"{stem}.jpg"\n            lbl_path = lbl_dir / f"{stem}.txt"\n\n            lines: list[str] = []\n            for ann in _ann_list_from_row(row):\n                cid = int(ann.get("category_id", -1))\n                ours = COCO_TO_OURS.get(cid)\n                if ours is None:\n                    continue\n                seg = ann.get("segmentation") or []\n                for body in _poly_to_yolo_seg(seg, width, height):\n                    lines.append(f"{ours} {body}")\n\n            if not lines:\n                continue\n            image.convert("RGB").save(img_path, quality=92)\n            lbl_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n            counts[split_dst] += 1\n            if counts[split_dst] % 100 == 0:\n                print(f"  {split_dst}: {counts[split_dst]} images…")\n\n    write_data_yaml(out_dir)\n    (out_dir / "LICENSE.txt").write_text(LICENSE_NOTE + "\\n", encoding="utf-8")\n    return counts\n\n\ndef export_weak_labels(\n    out_dir: Path,\n    image_paths: Iterable[Path],\n    model_path: Path,\n    *,\n    split: str = "train",\n    conf: float = 0.25,\n    imgsz: int = 640,\n) -> int:\n    """Pseudo-label fixture images with a pretrained YOLO-seg and append to dataset."""\n    from ultralytics import YOLO\n    from PIL import Image\n\n    model = YOLO(str(model_path))\n    img_dir = out_dir / "images" / split\n    lbl_dir = out_dir / "labels" / split\n    img_dir.mkdir(parents=True, exist_ok=True)\n    lbl_dir.mkdir(parents=True, exist_ok=True)\n\n    name_to_id = {v: k for k, v in CLASS_NAMES.items()}\n    # also accept model.names strings\n    n = 0\n    for src in image_paths:\n        src = Path(src)\n        if not src.is_file():\n            continue\n        results = model.predict(str(src), conf=conf, imgsz=imgsz, verbose=False)\n        if not results:\n            continue\n        r0 = results[0]\n        h, w = int(r0.orig_shape[0]), int(r0.orig_shape[1])\n        lines: list[str] = []\n        if r0.masks is not None and r0.boxes is not None:\n            cls_ids = r0.boxes.cls.cpu().numpy().astype(int)\n            # ultralytics masks.xyn is already normalized polygons\n            xyn = r0.masks.xyn\n            model_names = r0.names or {}\n            for ci, poly in zip(cls_ids, xyn):\n                cname = str(model_names.get(int(ci), "")).lower()\n                ours = name_to_id.get(cname)\n                if ours is None:\n                    # try alias\n                    if cname in ("rooms", "area", "space"):\n                        ours = 0\n                    elif cname in ("walls",):\n                        ours = 1\n                    elif cname in ("doors", "opening"):\n                        ours = 2\n                    elif cname in ("windows", "win"):\n                        ours = 3\n                    elif cname in ("stairs",):\n                        ours = 4\n                    else:\n                        continue\n                flat = poly.reshape(-1)\n                if flat.size < 6:\n                    continue\n                coords = " ".join(f"{float(v):.6f}" for v in flat)\n                lines.append(f"{ours} {coords}")\n        if not lines:\n            continue\n        stem = f"weak_{src.stem}"\n        dst = img_dir / f"{stem}{src.suffix.lower() if src.suffix else \'.jpg\'}"\n        if src.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:\n            shutil.copy2(src, dst)\n        else:\n            Image.open(src).convert("RGB").save(dst.with_suffix(".jpg"), quality=92)\n            dst = dst.with_suffix(".jpg")\n        (lbl_dir / f"{dst.stem}.txt").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n        n += 1\n        print(f"  weak-label: {src.name} → {len(lines)} instances")\n    write_data_yaml(out_dir)\n    return n\n\n\ndef make_synthetic_smoke(out_dir: Path, n_train: int = 24, n_val: int = 6) -> dict[str, int]:\n    """Tiny synthetic rectangles so the training loop can be smoke-tested offline."""\n    import numpy as np\n    from PIL import Image, ImageDraw\n\n    counts = {"train": 0, "val": 0}\n    rng = random.Random(0)\n    for split, n in (("train", n_train), ("val", n_val)):\n        img_dir = out_dir / "images" / split\n        lbl_dir = out_dir / "labels" / split\n        img_dir.mkdir(parents=True, exist_ok=True)\n        lbl_dir.mkdir(parents=True, exist_ok=True)\n        for i in range(n):\n            w, h = 640, 480\n            im = Image.new("RGB", (w, h), (245, 245, 240))\n            draw = ImageDraw.Draw(im)\n            lines: list[str] = []\n            # outer wall ring\n            margin = 40\n            wall_t = 12\n            draw.rectangle([margin, margin, w - margin, h - margin], outline=(30, 30, 30), width=wall_t)\n            # wall poly (outer rect as thin ring approximated by filled outer - skip; use box)\n            def box_poly(x0, y0, x1, y1, cls_id: int) -> None:\n                xs = [x0 / w, y0 / h, x1 / w, y0 / h, x1 / w, y1 / h, x0 / w, y1 / h]\n                lines.append(f"{cls_id} " + " ".join(f"{v:.6f}" for v in xs))\n\n            box_poly(margin, margin, w - margin, margin + wall_t, 1)\n            box_poly(margin, h - margin - wall_t, w - margin, h - margin, 1)\n            box_poly(margin, margin, margin + wall_t, h - margin, 1)\n            box_poly(w - margin - wall_t, margin, w - margin, h - margin, 1)\n            # rooms\n            mid = w // 2\n            box_poly(margin + wall_t + 4, margin + wall_t + 4, mid - 4, h - margin - wall_t - 4, 0)\n            box_poly(mid + 4, margin + wall_t + 4, w - margin - wall_t - 4, h - margin - wall_t - 4, 0)\n            # door / window\n            dx = mid - 20\n            box_poly(dx, h - margin - wall_t - 2, dx + 40, h - margin + 2, 2)\n            wx = margin + 80 + rng.randint(0, 40)\n            box_poly(wx, margin - 2, wx + 50, margin + wall_t + 2, 3)\n            stem = f"synth_{split}_{i:03d}"\n            im.save(img_dir / f"{stem}.jpg", quality=90)\n            (lbl_dir / f"{stem}.txt").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n            counts[split] += 1\n    write_data_yaml(out_dir)\n    (out_dir / "LICENSE.txt").write_text(\n        "Synthetic smoke data generated by prepare_yolo_dataset.py (public domain).\\n",\n        encoding="utf-8",\n    )\n    return counts\n\n\ndef main() -> int:\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--out", type=Path, default=Path("./fp_yolo_dataset"))\n    p.add_argument("--max-train", type=int, default=800, help="Cap CubiCasa train images (None=all via -1)")\n    p.add_argument("--max-val", type=int, default=80)\n    p.add_argument("--synthetic-only", action="store_true", help="Skip CubiCasa; write tiny synthetic set")\n    p.add_argument("--weak-label-dir", type=Path, default=None, help="Dir of *.jpg for pseudo-labels")\n    p.add_argument("--weak-model", type=Path, default=None, help="YOLO-seg .pt for weak labels")\n    p.add_argument("--seed", type=int, default=42)\n    args = p.parse_args()\n\n    out = args.out\n    out.mkdir(parents=True, exist_ok=True)\n\n    max_train = None if args.max_train is not None and args.max_train < 0 else args.max_train\n    max_val = None if args.max_val is not None and args.max_val < 0 else args.max_val\n\n    if args.synthetic_only:\n        counts = make_synthetic_smoke(out)\n        print(f"Synthetic dataset → {out} {counts}")\n    else:\n        try:\n            counts = export_cubicasa_hf(out, max_train=max_train, max_val=max_val, seed=args.seed)\n            print(f"CubiCasa export → {out} {counts}")\n        except SystemExit:\n            raise\n        except Exception as e:  # noqa: BLE001\n            print(f"CubiCasa download/convert failed: {e}", file=sys.stderr)\n            print("Falling back to synthetic smoke dataset…", file=sys.stderr)\n            counts = make_synthetic_smoke(out)\n            print(f"Synthetic fallback → {out} {counts}")\n\n    if args.weak_label_dir and args.weak_model:\n        paths = sorted(args.weak_label_dir.glob("*-b.jpg"))\n        if not paths:\n            paths = sorted(args.weak_label_dir.glob("*.jpg"))\n        n = export_weak_labels(out, paths, args.weak_model, split="train")\n        print(f"Added {n} weak-labeled images from {args.weak_label_dir}")\n\n    meta = {\n        "out": str(out.resolve()),\n        "counts": counts if isinstance(counts, dict) else {},\n        "license": LICENSE_NOTE,\n        "class_names": CLASS_NAMES,\n    }\n    (out / "prepare_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    print(f"Wrote {out / \'data.yaml\'}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
print('embedded prepare bytes', len(EMBEDDED_PREPARE))


In [ ]:
# ===== 3) Ensure prepare_yolo_dataset.py (self-contained embed) =====
from pathlib import Path

dest = WORK / "prepare_yolo_dataset.py"
found = next((p for p in PREPARE_SRC_CANDIDATES if p.is_file()), None)
if found:
    dest.write_bytes(found.read_bytes())
    print("copied", found, "→", dest)
else:
    # Embedded copy from repo backend/training/prepare_yolo_dataset.py
    dest.write_text(EMBEDDED_PREPARE, encoding="utf-8")
    print("wrote embedded prepare_yolo_dataset.py →", dest)

print("prepare script:", dest, "bytes", dest.stat().st_size)


In [ ]:
# ===== 4) Optional: upload marketing/CAD weak-label images =====
# Colab: use left sidebar Files to upload 2b69218a-b.jpg, 964226aa-b.jpg, bb00b5bf-b.jpg
# into WEAK_DIR. Or skip — CubiCasa alone still trains.
WEAK_DIR = WORK / "weak_images"
WEAK_DIR.mkdir(parents=True, exist_ok=True)

# Also search common upload locations
from pathlib import Path
import shutil
extra_roots = [Path("/content"), Path("/kaggle/input"), Path(".")]
for root in extra_roots:
    if not root.exists():
        continue
    for p in root.rglob("*-b.jpg"):
        if "weak_images" in str(p):
            continue
        target = WEAK_DIR / p.name
        if not target.exists():
            try:
                shutil.copy2(p, target)
                print("staged weak image", p.name)
            except Exception:
                pass

print("weak images:", list(WEAK_DIR.glob("*.jpg")))


In [ ]:
# ===== 5) Build YOLO dataset =====
import subprocess, sys
from pathlib import Path

DATA_OUT = WORK / "fp_yolo_dataset"
cmd = [
    sys.executable, str(WORK / "prepare_yolo_dataset.py"),
    "--out", str(DATA_OUT),
    "--max-train", str(MAX_TRAIN),
    "--max-val", str(MAX_VAL),
]
if not USE_CUBICASA:
    cmd.append("--synthetic-only")

# Base weight for weak labels (download RW-seg)
from huggingface_hub import hf_hub_download
weak_model = None
if USE_WEAK_LABELS and any(WEAK_DIR.glob("*.jpg")):
    try:
        weak_model = Path(hf_hub_download("JessiP23/floorplan-seg-v2", "best.pt"))
        cmd += ["--weak-label-dir", str(WEAK_DIR), "--weak-model", str(weak_model)]
    except Exception as e:
        print("weak model download failed, skip weak labels:", e)

print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print((DATA_OUT / "data.yaml").read_text())
print((DATA_OUT / "LICENSE.txt").read_text() if (DATA_OUT / "LICENSE.txt").exists() else "")


In [ ]:
# ===== 6) Choose imgsz from VRAM =====
import torch

imgsz = IMGSZ
batch = BATCH
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 14 and IMGSZ < 896:
        imgsz = 896
        batch = max(2, BATCH // 2)
        print(f"VRAM ~{vram:.1f}GB → try imgsz={imgsz} batch={batch}")
    elif vram < 10:
        imgsz = min(imgsz, 640)
        batch = min(batch, 4)
        print(f"Low VRAM ~{vram:.1f}GB → imgsz={imgsz} batch={batch}")
else:
    imgsz = 416
    batch = 2
    print("CPU fallback imgsz=416")

IMGSZ_EFF, BATCH_EFF = imgsz, batch


In [ ]:
# ===== 7) Load base model =====
from pathlib import Path
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

base_pt = None
try:
    base_pt = Path(hf_hub_download("JessiP23/floorplan-seg-v2", "best.pt"))
    print("base: JessiP23/floorplan-seg-v2", base_pt)
except Exception as e:
    print("HF base failed, using ultralytics yolo11s-seg.pt:", e)
    base_pt = "yolo11s-seg.pt"

model = YOLO(str(base_pt))
print("names", model.names)


In [ ]:
# ===== 8) Train =====
from ultralytics import YOLO
import torch

data_yaml = str(DATA_OUT / "data.yaml")
device = 0 if torch.cuda.is_available() else "cpu"

train_kwargs = dict(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ_EFF,
    batch=BATCH_EFF,
    device=device,
    project=str(WORK / PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
    workers=2,
    resume=RESUME,
)

# OOM retry ladder
attempts = [
    dict(train_kwargs),
    {**train_kwargs, "batch": max(1, BATCH_EFF // 2), "imgsz": min(IMGSZ_EFF, 640)},
    {**train_kwargs, "batch": 1, "imgsz": 640},
]

last_err = None
results = None
for i, kw in enumerate(attempts):
    try:
        print(f"train attempt {i+1}: imgsz={kw['imgsz']} batch={kw['batch']}")
        results = model.train(**kw)
        break
    except RuntimeError as e:
        last_err = e
        if "out of memory" in str(e).lower() or "CUDA" in str(e):
            print("OOM → retry with smaller settings")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # reload model cleanly
            model = YOLO(str(base_pt))
            continue
        raise

if results is None:
    raise last_err or RuntimeError("training failed")

print("training finished")


In [ ]:
# ===== 9) Locate best.pt + show metrics =====
from pathlib import Path
import shutil

run_dir = WORK / PROJECT / RUN_NAME
best = run_dir / "weights" / "best.pt"
last = run_dir / "weights" / "last.pt"
print("run_dir", run_dir)
print("best exists", best.is_file(), best)
print("last exists", last.is_file(), last)

# Convenience copies at WORK root for easy Colab download
if best.is_file():
    shutil.copy2(best, WORK / "best.pt")
    shutil.copy2(best, WORK / "floorplan-rw-seg.pt")
    print("Copied to", WORK / "best.pt")
    print("SIZE_BYTES", best.stat().st_size)

results_csv = run_dir / "results.csv"
if results_csv.is_file():
    print(results_csv.read_text().splitlines()[-3:])


## 10) Download weights → local backend

**Colab:** left sidebar → `/content/fp_train/best.pt` → Download.

**Kaggle:** Output → `fp_train/best.pt`.

On your machine (repo checkout):

```bash
cd backend
source .venv/bin/activate
python scripts/install_finetuned_weight.py ~/Downloads/best.pt
export DETECT_MODE=yolo
uvicorn main:app --reload --host 127.0.0.1 --port 8000
python scripts/regress_detect.py
```

Or set a custom path:

```bash
export DETECT_MODEL_PATH=/absolute/path/to/floorplan-rw-seg.pt
export DETECT_MODE=yolo
```

See `backend/training/README.md` and `backend/fixtures/QA-BLOCKER-gpu.md`.


In [ ]:
# ===== 11) Zip for download (optional) =====
from pathlib import Path
import shutil
zip_base = WORK / "floorplan_seg_best"
if (WORK / "best.pt").is_file():
    # zip single file via temporary folder
    tmp = WORK / "_pack"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    shutil.copy2(WORK / "best.pt", tmp / "best.pt")
    shutil.copy2(WORK / "best.pt", tmp / "floorplan-rw-seg.pt")
    readme = tmp / "INSTALL.txt"
    readme.write_text(
        "Copy best.pt into home-planner/backend/models/ as floorplan-rw-seg.pt\n"
        "Or: python scripts/install_finetuned_weight.py best.pt\n"
        "Then: export DETECT_MODE=yolo\n"
        "CubiCasa5K labels used under CC BY-NC 4.0 (non-commercial).\n"
    )
    out = shutil.make_archive(str(zip_base), "zip", root_dir=tmp)
    print("ZIP:", out)
else:
    print("No best.pt yet — run training cell first")
